# 00 — Verify Connection (Zerobus Ingest)

Confirms Databricks Connect is working and that the Zerobus Ingest SDK is
installed. Zerobus requires a **service principal** for authentication —
you'll need the client ID and client secret in your `.env` file.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()
print("Spark version:", spark.version)

In [ ]:
spark.sql("SELECT current_user(), current_timestamp()").show(truncate=False)

## Verify Zerobus SDK is installed

In [ ]:
try:
    from zerobus.sdk.sync import ZerobusSdk
    from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
    print("Zerobus SDK imported successfully.")
except ImportError:
    print("ERROR: Zerobus SDK not installed.")
    print("Run: pip install databricks-zerobus-ingest-sdk")

## Verify Zerobus environment variables

Zerobus needs three additional env vars beyond standard Databricks Connect:
- `ZEROBUS_SERVER_ENDPOINT` — `https://{workspace_id}.zerobus.{region}.cloud.databricks.com`
- `ZEROBUS_CLIENT_ID` — Service principal client ID
- `ZEROBUS_CLIENT_SECRET` — Service principal client secret

To create a service principal:
1. Go to **Settings → Identity and Access → Service principals**
2. Click **Add service principal** → generate OAuth secret
3. Grant it `USE CATALOG`, `USE SCHEMA`, `SELECT`, and `MODIFY` on your target table

In [ ]:
required_vars = ["ZEROBUS_SERVER_ENDPOINT", "ZEROBUS_CLIENT_ID", "ZEROBUS_CLIENT_SECRET"]

for var in required_vars:
    val = os.getenv(var)
    if val:
        display_val = val[:20] + "..." if len(val) > 20 else val
        print(f"  {var}: {display_val}")
    else:
        print(f"  {var}: *** MISSING — check your .env file ***")

In [ ]:
UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")
print(f"Target: {UC_CATALOG}.{UC_SCHEMA}")